# BFCN

**Banana Flavored Config Notation** is file format standard, that:
- compatible with json,
  - every JSON **is** valid BFCN
  - every BFCN file **can be converted** to valid JSON


## Goal
Shift JSON more into the direction of config file, rather than data-transfer
standard. Here are the main changes and their rationles:

### Comments

Added comments, denoted as `//`. Example:
```
{
    // Here you have to add relative path to the file
    "input_path": "./filename.txt", 
    // number of threads used
    "kthreads": 4
}
```
**Rationale**: Lack of the comments in JSON standard can be related to the fact to heavy-use as a data-transfer protocol. In opposite to it, in config files we don't have to care that much about it. Conversion to JSON can be easily done after parsing.

### Trailing comma

#### Feature test

Before betting on the dead horse, I will check if it's possible to parse JSON grammar with trailing commas in arrays and objects. I will check first with LR(0) and SLR parser.

In [ ]:
value = ["V -> number", "V -> string", "V -> bool", "V -> null", "V -> L", "V -> O"]

object_rules = [
    "O -> { Pair LastPair }",
    "Pair -> string : V",
    "LastPair -> , Pair LastPair",
    "LastPair -> ,",
    "LastPair -> ",
]
list_rules = [
    "L -> [ V LastElem ]",
    "LastElem -> , V LastElem",
    "LastElem -> ,",
    "LastElem -> ",
]
start = [
    "S -> V $",
]

bfcn_grammar = value + object_rules + list_rules + start

In [ ]:
from parsers.lr.lr0 import LR0Parser

In [ ]:
lr0 = LR0Parser(*bfcn_grammar)
print(lr0.to_tabulate())

In [ ]:
from collections import defaultdict

In [ ]:
conflicts = defaultdict(int)
for symbol in lr0.symbols:
    for row in lr0.parsing_table.values():
        actions = row[symbol]
        if len(actions) <= 1:
            continue
        kind = tuple(sorted([el.type_ for el in actions]))
        conflicts[kind] += 1

In [ ]:
conflicts

In [ ]:
from parsers.lr.slr import SLRParser

In [ ]:
slr = SLRParser(*bfcn_grammar)
print(slr.to_tabulate())

In [ ]:
conflicts = defaultdict(int)
for symbol in slr.symbols:
    for row in slr.parsing_table.values():
        actions = row[symbol]
        if len(actions) <= 1:
            continue
        kind = tuple(sorted([el.type_ for el in actions]))
        conflicts[kind] += 1

In [ ]:
conflicts

No conflicts in SLR parser, so it should work! That's great news.

*Rationale*: This is minor change, but it makes bfcn easier to edit, manage and read. Should be easy to covert to json, just remove comma and you're good to go.

### First object without brackets

Probably as a config file format, most files in BFCN will have object as a first type in them. Removing the need for first object to have brackets could make this process way easier.

In [ ]:
no_brackets_object_rules = [
    "O -> { Object }",
    "Object -> Pair LastPair",
    "Pair -> string : V",
    "LastPair -> , Pair LastPair",
    "LastPair -> ,",
    "LastPair -> ",
]

no_brackets_start = [
    "S -> S` $",
    "S` -> V",
    "S` -> Object",
]

no_brackets_grammar = value + no_brackets_object_rules + list_rules + no_brackets_start

In [ ]:
no_brackets_slr = SLRParser(*no_brackets_grammar)
print(no_brackets_slr.to_tabulate())

In [ ]:
conflicts = defaultdict(int)
for symbol in no_brackets_slr.symbols:
    for row in no_brackets_slr.parsing_table.values():
        actions = row[symbol]
        if len(actions) <= 1:
            continue
        kind = tuple(sorted([el.type_ for el in actions]))
        conflicts[kind] += 1
conflicts

No conflict was found, probably this grammar can be easily parsed using SLR.